## Location Song Data Collection Notebook

This notebook contains the codes for retrieving songs from Spotify API. As stated in my dissertation report, there is need for location, age and gender context aware recommendation system. This notebook does data collection of location song. It also gets the songs metadata - artist_name, track_name, album, duration_ms, etc. The other musical features that were added after the interview with participant is not in this notebook - It can be found in Musical Features Data Collection Notebook. The genre collection which took a different approach is found in Genre Collection Notebook.

In [ ]:
# Install Spotipy library
!pip install spotipy
!pip install lyricsgenius # this was not used in this notebook

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.1/252.1 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 kB 1.9 MB/s eta 0:00:00


In [ ]:
# Import Library
import pandas as pd
import numpy as np
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

In [ ]:
# Songs in each album using Spotify

SPOTIPY_CLIENT_ID='YOUR_SPOTIFY_CLIENT_ID'
SPOTIPY_CLIENT_SECRET='YOUR_SPOTIFY_CLIENT_SECRET'

# Initialize Spotify client credentials manager
auth_manager = SpotifyClientCredentials(client_id=SPOTIPY_CLIENT_ID, client_secret=SPOTIPY_CLIENT_SECRET)

# Create a Spotify client
sp = spotipy.Spotify(auth_manager=auth_manager)

In [ ]:
# Fetches the spotify playlists available in my account region
playlists = sp.user_playlists('spotify')

# Loop continues while there are more playlists to process
while playlists:
    # Iterates through each playlist in the current 'page' of playlists
    for i, playlist in enumerate(playlists['items']):
        # Print the index (starting from 1 + offset), the playlist's URI, and its name
        print("%4d %s %s" % (i + 1 + playlists['offset'], playlist['uri'], playlist['name']))

    # Check if there is another page of playlists to fetch
    if playlists['next']:
        # Fetch the next page of playlists
        playlists = sp.next(playlists)
    else:
        # If there are no more pages, end the loop
        playlists = None

## After I got the list of spotify playlist, I manually checked playlist that are location (activity) labelled and created a dataframe with its name and plalist id.

In [ ]:
# create a dictionary named data that categorizes different playlists by their specific location
data = {'Gym': ['37i9dQZF1DWUVpAXiEPK8P', '37i9dQZF1DX0HRj9P7NxeE', '7s0efM9c1ZS427tw2B6wEF',
                '37i9dQZF1DXe6bgV3TmZOL', '37i9dQZF1DWSJHnPb1f0X3', '37i9dQZF1DX70RN3TfWWJh',
                '37i9dQZF1DWZqUHC2tviPw', '2YeGkAUQhmO8TCjSMbFYWf', '37i9dQZF1DXdxcBWuJkbcy'],
       'Bedroom': ['37i9dQZF1DWSrVdvTl1tVY', '3NJt9AdW0HpyPTe8WP5XRi', '7HtQTTOI7WuNiX07YcTaTQ',
                   '0RTz1jFo5BXGPfI8eVf8sj', '3NJt9AdW0HpyPTe8WP5XRi', '7HtQTTOI7WuNiX07YcTaTQ',
                   '37i9dQZF1DXa8NOEUWPn9W', '37i9dQZF1DX6VdMW310YC7', '37i9dQZF1DX2TRYkJECvfC',
                   '37i9dQZF1DX6tTW0xDxScH'],
       'Library': ['37i9dQZF1DX8NTLI2TtZa6', '5TKYF80bhbhiHSsKS0IPyV',
                   '37i9dQZF1DWXLeA8Omikj7', '37i9dQZF1DWVcV87ePAqAq', '37i9dQZF1DWSluGMsH1R9r'],
       'Office': ['1oDFttzXiLUJLjMyulUugY', '02bG2SzzHkBk8CwWSpPUEL',
                  '00P3uFAffQkFRNeWd5BJgK', '3OFdlWHturvecA5qFGSRZ3', '1BY3FaXY7Lgek0Eh5kqPdt',
                  '37i9dQZF1DWTLSN7iG21yC', '6MtNKlWguIlTJ6Q7WAcDtU', '37i9dQZF1DXcsT4WKI8W8r'],
       'Train/Bus': ['0mY9BQvlpraYLzg9BfKuVa', '0NaXa68Xyo4wmMiun5Lqgm', '37i9dQZF1DX2MyUCsl25eb',
                     '37i9dQZF1DX3bSdu6sAEDF', '37i9dQZF1DWWMOmoXKqHTD', '37i9dQZF1DX0sQWfevMRw3'],
       'Restaurant': ['4lgseztVwmKQ8MNETPVIny', '1rq0xyAv9fuHC1zIuGsUM7', '06FioTm0pVsjoKvEUS6fxb',
                      '37i9dQZF1DX4s3V2rTswzO', '37i9dQZF1DX6ziVCJnEm59', '37i9dQZF1DXbm6HfkbMtFZ',
                      '37i9dQZF1DX4xuWVBs4FgJ', '37i9dQZF1DXab8DipvnuNU'],
       'Bar': ['4hTnMuT5xPK7uerXJyKYDs', '5mYPnjPRq0HkxT5MOkQMY4', '5LHlvHK1AFXLQKFpWEv4JP',
               '54qaOhKo54TWrhdRNneKn2', '37i9dQZF1DX4wta20PHgwo'],
       'Mall': ['2P0VBAcUPSzjhgYUsIyjjb', '6JlaZBTWBY95xuyIl1gZQm', '37i9dQZF1DX9pryhDLql25']}

In [ ]:
# name of the playlist used
playlist_name = ['Power Workout', 'Workout Twerkout', 'Top Tracks in Workout', 'Adrenaline Workout', 'Cardio',
                 'Workout', 'Trap Workout Beats', 'Energy + Positivity', 'Gym Hits',
                 'UK House Music', 'bedroom jams', 'house playlist', 'Top Bedroom Songs', 'bedroom jams',
                 'bedroom playlist', 'Bedroom Housewerk', 'Chill Bedroom Tracks', 'Deep Bedroom Relax', 'Stay in Bedroom',
                 'Library Intense Studying', 'library studying playlist', 'Library Brain Food', 'Library Studentfest!',
                 'Library Perfect Concentration',
                 'office Workday – Zen', 'office playlist', 'office playlist',
                 'office afrobeat', 'office playlist new', 'Office Work From Home', 'Office Workday - Country',
                 'Office Music for a Workday',
                 'Traffic Jams', 'Carpool Karaoke', 'Morning Commute', 'Evening Commute',
                 'Songs to Sing in the Bus', '30 km/h Commute',
                 'foodora dinner playlist', 'restuarant playlist', 'restuarant music', 'restuarant Romance',
                 'Your Favorite Restaurant', 'restuarant Dinner', 'restuarant with Friends', 'The Cookout',
                 'bar playlist', 'bar playlist', 'bar updated playlist', 'bar playlist', 'Late Night Bar Jazz',
                 'mall shopping music 2024', 'shopping music', 'Mall Music']

In [ ]:
# Initialize an empty list to store rows of data
rows = []

# Iterate through the dictionary data, extracting location and playlist IDs
for location, playlist_ids in data.items():
    for playlist_id in playlist_ids:
        # Append each (location, playlist_id) pair to the list as a tuple
        rows.append((location, playlist_id))

# Create a DataFrame from the list of tuples 'rows'
# Each row in the DataFrame represents a playlist along with its associated location
playlist = pd.DataFrame(rows, columns=['location', 'playlist_id'])

# Add a new column 'playlist_name' to the DataFrame
playlist['playlist_name'] = playlist_name


For all the location in the subsequent cell, the code first initializes an empty list to store track data from the location playlists. It then iterates over the a certain number of playlistid which is categorized as that locations playlists, using Spotify's API to fetch up to 100 tracks per playlist. The resulting track data is appended to the empty list. Next, the code processes each track in the list to extract key details—such as track name, artist name, release date, popularity, duration, and track ID—and stores them in a new list, playlist_data_'location'. Finally, this data is used to create a pandas DataFrame named the location eg Gym, which organizes all the extracted track information into a structured tabular format for further analysis.

In [ ]:
# Initialize an empty list to store tracks from the gym playlists
tracks_Gym = []

# Loop through the first 9 playlist IDs from the 'playlist_id' column in the DataFrame which are gym playlists
for playlist_id in playlist['playlist_id'][0:9]:
    # Fetch all the tracks from the specified playlist using Spotify API
    # The playlist ID is taken from the 'Gym' category and retrieves up to 100 tracks starting from offset 0
    playlist_tracks_Gym = sp.user_playlist_tracks(user='316safnczqmimwhx4ea67xfgzi2m',
                                                  playlist_id=playlist_id,
                                                  fields=None,
                                                  limit=100,
                                                  offset=0,
                                                  market=None)
    # Append the fetched playlist tracks data to the list of gym tracks
    tracks_Gym.append(playlist_tracks_Gym)

# Initialize an empty list to store extracted track details for gym playlists
playlist_data_Gym = []

# Loop through each set of playlist tracks in 'tracks_Gym'
for track_list in tracks_Gym:
    # Loop through each track in the current playlist
    for track in track_list['items']:
        # Extract relevant track details: name, artist, release date, popularity, duration, and ID
        track_name = track['track']['name']
        artist_name = track['track']['artists'][0]['name']
        release_date = track['track']['album']['release_date']
        popularity = track['track']['popularity']
        duration_ms = track['track']['duration_ms']
        track_id = track['track']['id']

        # Append the track details to the 'playlist_data_Gym' list as a dictionary
        playlist_data_Gym.append({
            'track_name': track_name,
            'artist_name': artist_name,
            'release_date': release_date,
            'popularity': popularity,
            'duration_ms': duration_ms,
            'track_id': track_id})

# Create a DataFrame from the list of track details for gym playlists
Gym = pd.DataFrame(playlist_data_Gym)


In [ ]:
# Initialize an empty list to store tracks from the library playlists
tracks_Library = []

# Loop through the next set of playlist IDs (indices 19 to 23) from the 'playlist_id' column in the DataFrame which are library playlists
for playlist_id in playlist['playlist_id'][19:24]:
    # Fetch all the tracks from the specified playlist using Spotify API
    # The playlist ID is taken from the 'Library' category and retrieves up to 100 tracks starting from offset 0
    playlist_tracks_Library = sp.user_playlist_tracks(user='316safnczqmimwhx4ea67xfgzi2m',
                                                      playlist_id=playlist_id,
                                                      fields=None,
                                                      limit=100,
                                                      offset=0,
                                                      market=None)
    # Append the fetched playlist tracks data to the list of library tracks
    tracks_Library.append(playlist_tracks_Library)

# Initialize an empty list to store extracted track details for library playlists
playlist_data_Library = []

# Loop through each set of playlist tracks in 'tracks_Library'
for track_list in tracks_Library:
    # Loop through each track in the current playlist
    for track in track_list['items']:
        # Extract relevant track details: name, artist, release date, popularity, duration, and ID
        track_name = track['track']['name']
        artist_name = track['track']['artists'][0]['name']
        release_date = track['track']['album']['release_date']
        popularity = track['track']['popularity']
        duration_ms = track['track']['duration_ms']
        track_id = track['track']['id']

        # Append the track details to the 'playlist_data_Library' list as a dictionary
        playlist_data_Library.append({
            'track_name': track_name,
            'artist_name': artist_name,
            'release_date': release_date,
            'popularity': popularity,
            'duration_ms': duration_ms,
            'track_id': track_id})

# Create a DataFrame from the list of track details for library playlists
Library = pd.DataFrame(playlist_data_Library)

In [ ]:
# Initialize an empty list to store tracks from the office playlists
tracks_Office = []

# Loop through the playlist IDs from index 24 to 31 in the 'playlist_id' column, which are categorized as office playlists
for playlist_id in playlist['playlist_id'][24:32]:
    # Fetch all the tracks from the specified playlist using Spotify API
    # - Retrieve up to 100 tracks starting from offset 0
    playlist_tracks_Office = sp.user_playlist_tracks(user='316safnczqmimwhx4ea67xfgzi2m',
                                                     playlist_id=playlist_id,
                                                     fields=None,
                                                     limit=100,
                                                     offset=0,
                                                     market=None)
    # Append the fetched playlist tracks data to the list of office tracks
    tracks_Office.append(playlist_tracks_Office)

# Initialize an empty list to store extracted track details for office playlists
playlist_data_Office = []

# Loop through each set of playlist tracks in 'tracks_Office'
for track_list in tracks_Office:
    # Loop through each track in the current playlist
    for track in track_list['items']:
        # Extract relevant track details: name, artist, release date, popularity, duration, and ID
        track_name = track['track']['name']
        artist_name = track['track']['artists'][0]['name']
        release_date = track['track']['album']['release_date']
        popularity = track['track']['popularity']
        duration_ms = track['track']['duration_ms']
        track_id = track['track']['id']

        # Append the track details to the 'playlist_data_Office' list as a dictionary
        playlist_data_Office.append({
            'track_name': track_name,
            'artist_name': artist_name,
            'release_date': release_date,
            'popularity': popularity,
            'duration_ms': duration_ms,
            'track_id': track_id})

# Create a DataFrame from the list of track details for office playlists
Office = pd.DataFrame(playlist_data_Office)


In [ ]:
# Initialize an empty list to store tracks from the Train/Bus playlists
tracks_TrainBus = []

# Loop through the playlist IDs from index 32 to 37 in the 'playlist_id' column, which are categorized as Train/Bus playlists
for playlist_id in playlist['playlist_id'][32:38]:
    # Fetch all the tracks from the specified playlist using Spotify API
    # Retrieve up to 100 tracks starting from offset 0
    playlist_tracks_TrainBus = sp.user_playlist_tracks(user='316safnczqmimwhx4ea67xfgzi2m',
                                                       playlist_id=playlist_id,
                                                       fields=None,
                                                       limit=100,
                                                       offset=0,
                                                       market=None)
    # Append the fetched playlist tracks data to the list of Train/Bus tracks
    tracks_TrainBus.append(playlist_tracks_TrainBus)

# Initialize an empty list to store extracted track details for Train/Bus playlists
playlist_data_TrainBus = []

# Loop through each set of playlist tracks in 'tracks_TrainBus'
for track_list in tracks_TrainBus:
    # Loop through each track in the current playlist
    for track in track_list['items']:
        # Extract relevant track details: name, artist, release date, popularity, duration, and ID
        track_name = track['track']['name']
        artist_name = track['track']['artists'][0]['name']
        release_date = track['track']['album']['release_date']
        popularity = track['track']['popularity']
        duration_ms = track['track']['duration_ms']
        track_id = track['track']['id']

        # Append the track details to the 'playlist_data_TrainBus' list as a dictionary
        playlist_data_TrainBus.append({
            'track_name': track_name,
            'artist_name': artist_name,
            'release_date': release_date,
            'popularity': popularity,
            'duration_ms': duration_ms,
            'track_id': track_id})

# Create a DataFrame from the list of track details for Train/Bus playlists
TrainBus = pd.DataFrame(playlist_data_TrainBus)


In [ ]:
# Initialize an empty list to store tracks from the restaurant playlists
tracks_Restaurant = []

# Loop through the playlist IDs from index 38 to 45 in the 'playlist_id' column, which are categorized as restaurant playlists
for playlist_id in playlist['playlist_id'][38:46]:
    # Fetch all the tracks from the specified playlist using Spotify API
    # Retrieve up to 100 tracks starting from offset 0
    playlist_tracks_Restaurant = sp.user_playlist_tracks(user='316safnczqmimwhx4ea67xfgzi2m',
                                                         playlist_id=playlist_id,
                                                         fields=None,
                                                         limit=100,
                                                         offset=0,
                                                         market=None)
    # Append the fetched playlist tracks data to the list of restaurant tracks
    tracks_Restaurant.append(playlist_tracks_Restaurant)

# Initialize an empty list to store extracted track details for restaurant playlists
playlist_data_Restaurant = []

# Loop through each set of playlist tracks in 'tracks_Restaurant'
for track_list in tracks_Restaurant:
    # Loop through each track in the current playlist
    for track in track_list['items']:
        # Extract relevant track details: name, artist, release date, popularity, duration, and ID
        track_name = track['track']['name']
        artist_name = track['track']['artists'][0]['name']
        release_date = track['track']['album']['release_date']
        popularity = track['track']['popularity']
        duration_ms = track['track']['duration_ms']
        track_id = track['track']['id']

        # Append the track details to the 'playlist_data_Restaurant' list as a dictionary
        playlist_data_Restaurant.append({
            'track_name': track_name,
            'artist_name': artist_name,
            'release_date': release_date,
            'popularity': popularity,
            'duration_ms': duration_ms,
            'track_id': track_id})

# Create a DataFrame from the list of track details for restaurant playlists
Restaurant = pd.DataFrame(playlist_data_Restaurant)


In [ ]:
# Initialize an empty list to store tracks from the bar playlists
tracks_Bar = []

# Loop through the playlist IDs from index 46 to 50 in the 'playlist_id' column, which are categorized as bar playlists
for playlist_id in playlist['playlist_id'][46:51]:
    # Fetch all the tracks from the specified playlist using Spotify API
    # Retrieve up to 100 tracks starting from offset 0
    playlist_tracks_Bar = sp.user_playlist_tracks(user='316safnczqmimwhx4ea67xfgzi2m',
                                                  playlist_id=playlist_id,
                                                  fields=None,
                                                  limit=100,
                                                  offset=0,
                                                  market=None)
    # Append the fetched playlist tracks data to the list of bar tracks
    tracks_Bar.append(playlist_tracks_Bar)

# Initialize an empty list to store extracted track details for bar playlists
playlist_data_Bar = []

# Loop through each set of playlist tracks in 'tracks_Bar'
for track_list in tracks_Bar:
    # Loop through each track in the current playlist
    for track in track_list['items']:
        # Extract relevant track details: name, artist, release date, popularity, duration, and ID
        track_name = track['track']['name']
        artist_name = track['track']['artists'][0]['name']
        release_date = track['track']['album']['release_date']
        popularity = track['track']['popularity']
        duration_ms = track['track']['duration_ms']
        track_id = track['track']['id']

        # Append the track details to the 'playlist_data_Bar' list as a dictionary
        playlist_data_Bar.append({
            'track_name': track_name,
            'artist_name': artist_name,
            'release_date': release_date,
            'popularity': popularity,
            'duration_ms': duration_ms,
            'track_id': track_id})

# Create a DataFrame from the list of track details for bar playlists
Bar = pd.DataFrame(playlist_data_Bar)

In [ ]:
# Initialize an empty list to store tracks from the mall playlists
tracks_Mall = []

# Loop through all remaining playlist IDs from index 51 onwards in the 'playlist_id' column, categorized as mall playlists
for playlist_id in playlist['playlist_id'][51:]:
    # Fetch all the tracks from the specified playlist using Spotify API
    # Retrieve up to 100 tracks starting from offset 0 (first 100 tracks of the playlist)
    playlist_tracks_Mall = sp.user_playlist_tracks(user='316safnczqmimwhx4ea67xfgzi2m',
                                                   playlist_id=playlist_id,
                                                   fields=None,
                                                   limit=100,
                                                   offset=0,
                                                   market=None)
    # Append the fetched playlist tracks data to the list of mall tracks
    tracks_Mall.append(playlist_tracks_Mall)

# Initialize an empty list to store extracted track details for mall playlists
playlist_data_Mall = []

# Loop through each set of playlist tracks stored in 'tracks_Mall'
for track_list in tracks_Mall:
    # Loop through each track in the current playlist to extract relevant information
    for track in track_list['items']:
        # Extract track name
        track_name = track['track']['name']
        # Extract the name of the primary artist (first artist listed)
        artist_name = track['track']['artists'][0]['name']
        # Extract the album release date
        release_date = track['track']['album']['release_date']
        # Extract the track's popularity score
        popularity = track['track']['popularity']
        # Extract the track's duration in milliseconds
        duration_ms = track['track']['duration_ms']
        # Extract the Spotify ID for the track
        track_id = track['track']['id']

        # Append the extracted track details to the 'playlist_data_Mall' list as a dictionary
        playlist_data_Mall.append({
            'track_name': track_name,
            'artist_name': artist_name,
            'release_date': release_date,
            'popularity': popularity,
            'duration_ms': duration_ms,
            'track_id': track_id})

# Create a DataFrame from the list of track details for mall playlists
Mall = pd.DataFrame(playlist_data_Mall)


In [ ]:
# Initialize an empty list to store tracks from the bedroom playlists
tracks_Bedroom = []

# Loop through the playlist IDs from index 9 to 18 in the 'playlist_id' column, which are categorized as bedroom playlists
for playlist_id in playlist['playlist_id'][9:19]:
    # Fetch all the tracks from the specified playlist using Spotify API
    # Retrieves up to 100 tracks from the playlist starting from offset 0
    playlist_tracks_Bedroom = sp.user_playlist_tracks(user='316safnczqmimwhx4ea67xfgzi2m',
                                                      playlist_id=playlist_id,
                                                      fields=None,
                                                      limit=100,
                                                      offset=0,
                                                      market=None)
    # Append the fetched playlist tracks data to the list of bedroom tracks
    tracks_Bedroom.append(playlist_tracks_Bedroom)

# Initialize an empty list to store extracted track details for bedroom playlists
playlist_data_Bedroom = []

# Loop through each set of playlist tracks in 'tracks_Bedroom'
for track_list in tracks_Bedroom:
    # Loop through each track in the current playlist to extract relevant information
    for track in track_list['items']:
        # Add a check if 'track' is not None before accessing its elements to prevent errors
        if track and 'track' in track and track['track'] is not None:
            # Extract track details including name, artist, release date, popularity, duration, and ID
            track_name = track['track']['name']
            artist_name = track['track']['artists'][0]['name']
            release_date = track['track']['album']['release_date']
            popularity = track['track']['popularity']
            duration_ms = track['track']['duration_ms']
            track_id = track['track']['id']

            # Append the track details to the 'playlist_data_Bedroom' list as a dictionary
            playlist_data_Bedroom.append({
                'track_name': track_name,
                'artist_name': artist_name,
                'release_date': release_date,
                'popularity': popularity,
                'duration_ms': duration_ms,
                'track_id': track_id})
        else:
            # Log a warning if a track is missing or invalid
            print("Warning: Encountered a None value for track. Skipping.")

# Create a DataFrame from the list of track details for bedroom playlists
Bedroom = pd.DataFrame(playlist_data_Bedroom)

## Save the dataset in CSV and generate the questionnaire from Excel

In [ ]:
Gym.to_csv('Gym.csv', index=False)
Bedroom.to_csv('Bedroom.csv', index=False)
Library.to_csv('Library.csv', index=False)
Office.to_csv('Office.csv', index=False)
TrainBus.to_csv('TrainBus.csv', index=False)
Restaurant.to_csv('Restaurant.csv', index=False)
Bar.to_csv('Bar.csv', index=False)
Mall.to_csv('Mall.csv', index=False)